In [44]:
!pip install -q numpy scipy librosa unidecode inflect
!apt-get update -qq
!apt-get install -y libsndfile1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
0 upgraded, 0 newly installed, 0 to remove and 176 not upgraded.


In [45]:
import torch
import numpy as np
from scipy.io.wavfile import write
from IPython.display import Audio

In [46]:
tacotron2 = torch.hub.load(
    'NVIDIA/DeepLearningExamples:torchhub',
    'nvidia_tacotron2',
    model_math='fp32'
)

tacotron2 = tacotron2.cuda().eval()

Using cache found in /root/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub


In [47]:
waveglow = torch.hub.load(
    'NVIDIA/DeepLearningExamples:torchhub',
    'nvidia_waveglow',
    model_math='fp32'
)

waveglow = waveglow.remove_weightnorm(waveglow)
waveglow = waveglow.cuda().eval()

Using cache found in /root/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub


In [48]:
utils = torch.hub.load(
    'NVIDIA/DeepLearningExamples:torchhub',
    'nvidia_tts_utils'
)

Using cache found in /root/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub


In [49]:
text = """
 "Welcome back to the grand finale of the T20 Championship.",
    "The crowd is on its feet as the batting side needs eighteen runs from the final over.",
    "The bowler starts his run up.",
    "First ball. Short and pulled away. That's a boundary. Four runs.",
    "The equation now is fourteen needed from five balls."
"""

In [50]:
tacotron2.decoder.max_decoder_steps = 10000

In [55]:
sequences, lengths = utils.prepare_input_sequence([text])

sequences = sequences.cuda()
lengths = lengths.cuda()

with torch.no_grad():
    mel, _, _ = tacotron2.infer(sequences, lengths)
    audio = waveglow.infer(mel)

audio_numpy = audio[0].cpu().numpy()

In [56]:
sampling_rate = 22050

write(
    "output.wav",
    sampling_rate,
    audio_numpy
)

print("Saved as output.wav")

Saved as output.wav


In [57]:
Audio("output.wav")

In [ ]:
print(tacotron2.decoder.max_decoder_steps)